## **Daily Challenge : Building Trustworthy Insights with BERT**

### REPONSE


### 1. Chargement et inspection des donnees


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'transformers', 'scikit-learn', 'matplotlib', 'seaborn', 'scipy'])

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

dataset = load_dataset('tweet_eval', 'sentiment')

print('Repartition des donnees:')
for split in ['train', 'validation', 'test']:
    print(f'{split}: {len(dataset[split])} exemples')

labels_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
train_labels = [dataset['train'][i]['label'] for i in range(len(dataset['train']))]
label_counts = Counter(train_labels)
print('Repartition par classe (train):')
for label_id, count in sorted(label_counts.items()):
    print(f'  {labels_map[label_id]}: {count}')

examples_by_label = {0: [], 1: [], 2: []}
for i, item in enumerate(dataset['train']):
    if len(examples_by_label[item['label']]) < 2:
        examples_by_label[item['label']].append(item['text'])
    if all(len(v) >= 2 for v in examples_by_label.values()):
        break

print('Exemples par etiquette:')
for label_id, examples in examples_by_label.items():
    print(f'{labels_map[label_id]}:')
    for ex in examples:
        print(f'  - {ex[:100]}...')


### 2. Pipeline de tokenisation


In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['text', 'text_type', 'keyword', 'hashtags', 'ids']
)

tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
print(f'Dataset tokenise: {len(tokenized_dataset["train"])} echantillons')


### 3. Reglage fin


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model('./sentiment_model')
tokenizer.save_pretrained('./sentiment_model')
print('Modele sauvegarde dans ./sentiment_model')


### 4. Evaluation et etalonnage


In [ ]:
results = trainer.evaluate()
print(f'Resultats validation: {results}')

predictions = trainer.predict(tokenized_dataset['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

import scipy.special
softmax_scores = scipy.special.softmax(predictions.predictions, axis=1)
max_scores = np.max(softmax_scores, axis=1)

test_acc = accuracy_score(true_labels, pred_labels)
test_f1 = f1_score(true_labels, pred_labels, average='macro')

print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1 Macro: {test_f1:.4f}')

plt.figure(figsize=(10, 6))
plt.hist(max_scores, bins=10, range=(0, 1), edgecolor='black', alpha=0.7)
plt.xlabel('Score de confiance (softmax max)')
plt.ylabel('Nombre d exemples')
plt.title('Distribution des scores de confiance')
plt.grid(True, alpha=0.3)
plt.savefig('confidence_histogram.png', dpi=100, bbox_inches='tight')
plt.show()

mean_conf = np.mean(max_scores)
print(f'Score de confiance moyen: {mean_conf:.3f}')
if mean_conf > 0.8:
    print('Tendance: surestimation (surconfiance)')
elif mean_conf < 0.6:
    print('Tendance: sous-confiance')
else:
    print('Tendance: calibration moyenne')


### 5. Inspection d'attention


In [ ]:
attention_model = AutoModel.from_pretrained(model_name, output_attentions=True)

example_text = examples_by_label[0][0]
print(f'Tweet analyse: {example_text[:100]}...')

inputs = tokenizer(example_text, return_tensors='pt', truncation=True, max_length=128)

with torch.no_grad():
    outputs = attention_model(**inputs)

attentions = outputs.attentions[-1]
cls_attention = attentions.mean(dim=0)[0, 0, :].numpy()

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

plt.figure(figsize=(12, 6))
plt.bar(range(len(cls_attention)), cls_attention)
plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
plt.xlabel('Tokens')
plt.ylabel('Attention vers [CLS]')
plt.title('Carte d attention [CLS] - Tweet negatif')
plt.tight_layout()
plt.savefig('attention_cls_negative.png', dpi=100, bbox_inches='tight')
plt.show()

print('Tokens avec forte attention vers [CLS]:')
top_indices = cls_attention.argsort()[-5:][::-1]
for idx in top_indices:
    print(f'  {tokens[idx]}: {cls_attention[idx]:.3f}')


### 6. Fonction d'analyse pour production


In [ ]:
def analyze_text(text, model_path='./sentiment_model'):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()

    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
        logits = outputs.logits
        attentions = outputs.attentions[-1]

    probs = torch.nn.functional.softmax(logits, dim=-1)
    confidence, pred_label = torch.max(probs, dim=-1)
    pred_label = pred_label.item()
    confidence = confidence.item()

    cls_attention = attentions.mean(dim=0)[0, 0, :].numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    token_attention = [(t, a) for t, a in zip(tokens, cls_attention) 
                       if t not in ['[CLS]', '[SEP]', '[PAD]']]
    top_tokens = sorted(token_attention, key=lambda x: x[1], reverse=True)[:5]
    highlighted_tokens = [t for t, a in top_tokens]

    return {
        'label': ['negative', 'neutral', 'positive'][pred_label],
        'confidence': round(confidence, 3),
        'highlighted_tokens': highlighted_tokens
    }

result = analyze_text('I love this product, it works amazing!')
print(f'Resultat: {result}')

result_neg = analyze_text('This is terrible, worst service ever!')
print(f'Resultat negatif: {result_neg}')


## Livrable
Modele fine-tume, visualisations et fonction analyze_text() prets pour le deploiement.
